# Week 6 - Data Quality, Quarantine, Trusted Silver and Replay

This notebook implements P14-DQ-01 to P14-DQ-08,
routes Candidate records to Trusted Silver or Quarantine,
and demonstrates reconciliation and replay.

In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,bronze_complaints,false
default,bronze_departments,false
default,bronze_wards,false
default,silver_complaints,false
default,silver_departments,false
default,silver_wards,false


In [0]:
%sql

SELECT COUNT(*) AS candidate_rows
FROM cityfix.cityfix.silver_candidate_requests;

candidate_rows
180012


In [0]:
%sql

SELECT
    COUNT(*) AS candidate_rows,
    COUNT(DISTINCT physical_record_key) AS physical_keys,
    COUNT(DISTINCT unique_key) AS business_keys
FROM cityfix.cityfix.silver_candidate_requests;

candidate_rows,physical_keys,business_keys
180012,180012,179996


In [0]:
%sql

SELECT COUNT(*) AS candidate_rows
FROM cityfix.cityfix.silver_candidate_requests;

candidate_rows
180012


In [0]:
%sql

SELECT
    COUNT(*) AS candidate_rows,
    COUNT(DISTINCT physical_record_key) AS physical_keys,
    COUNT(DISTINCT unique_key) AS business_keys
FROM cityfix.cityfix.silver_candidate_requests;

candidate_rows,physical_keys,business_keys
180012,180012,179996


In [0]:
%sql

WITH duplicate_keys AS (
    SELECT
        unique_key
    FROM cityfix.cityfix.silver_candidate_requests
    WHERE unique_key IS NOT NULL
      AND TRIM(unique_key) <> ''
    GROUP BY unique_key
    HAVING COUNT(*) > 1
)

SELECT
    COUNT(*) AS duplicate_business_keys
FROM duplicate_keys;

duplicate_business_keys
12


In [0]:
%sql

SELECT
    COUNT(*) AS blank_or_null_unique_keys
FROM cityfix.cityfix.silver_candidate_requests
WHERE unique_key IS NULL
   OR TRIM(unique_key) = '';

blank_or_null_unique_keys
4


In [0]:
%sql

SELECT
    physical_record_key,
    unique_key,
    CASE
        WHEN unique_key IS NULL OR TRIM(unique_key) = '' THEN 'FAIL'
        WHEN COUNT(*) OVER (PARTITION BY unique_key) > 1 THEN 'FAIL'
        ELSE 'PASS'
    END AS P14_DQ_01
FROM cityfix.cityfix.silver_candidate_requests
LIMIT 100;

physical_record_key,unique_key,P14_DQ_01
REC-000002538,null,FAIL
REC-000075781,null,FAIL
REC-000151241,null,FAIL
REC-000177283,null,FAIL
REC-000000001,CFX-2025-000000001,PASS
REC-000000002,CFX-2025-000000002,PASS
REC-000000003,CFX-2025-000000003,PASS
REC-000000004,CFX-2025-000000004,PASS
REC-000000005,CFX-2025-000000005,PASS
REC-000000006,CFX-2025-000000006,PASS


In [0]:
%sql

SELECT
    physical_record_key,
    created_date,
    created_parse_status,

    CASE
        WHEN created_parse_status <> 'VALID'
            THEN 'FAIL'

        WHEN created_ts < CAST('2025-01-01' AS TIMESTAMP)
            THEN 'FAIL'

        WHEN created_ts >= CAST('2026-01-01' AS TIMESTAMP)
            THEN 'FAIL'

        ELSE 'PASS'
    END AS P14_DQ_02

FROM cityfix.cityfix.silver_candidate_requests

WHERE created_parse_status <> 'VALID'
   OR created_ts < CAST('2025-01-01' AS TIMESTAMP)
   OR created_ts >= CAST('2026-01-01' AS TIMESTAMP)

ORDER BY physical_record_key;

physical_record_key,created_date,created_parse_status,P14_DQ_02
REC-000003803,2024-12-15T10:00:00Z,VALID,FAIL
REC-000006673,2024-12-15T10:00:00Z,VALID,FAIL
REC-000026165,2024-12-15T10:00:00Z,VALID,FAIL
REC-000041988,not-a-timestamp,INVALID,FAIL
REC-000064382,2024-12-15T10:00:00Z,VALID,FAIL
REC-000065106,not-a-timestamp,INVALID,FAIL
REC-000067220,not-a-timestamp,INVALID,FAIL
REC-000079969,not-a-timestamp,INVALID,FAIL
REC-000094392,not-a-timestamp,INVALID,FAIL
REC-000097154,2024-12-15T10:00:00Z,VALID,FAIL


In [0]:
%sql

SELECT
    physical_record_key,
    created_date,
    due_date,
    created_parse_status,
    due_parse_status,

    CASE
        WHEN due_parse_status <> 'VALID'
            THEN 'FAIL'

        WHEN created_ts IS NOT NULL
             AND due_ts < created_ts
            THEN 'FAIL'

        ELSE 'PASS'
    END AS P14_DQ_03

FROM cityfix.cityfix.silver_candidate_requests

WHERE due_parse_status <> 'VALID'
   OR (
        created_ts IS NOT NULL
        AND due_ts < created_ts
      )

ORDER BY physical_record_key;

physical_record_key,created_date,due_date,created_parse_status,due_parse_status,P14_DQ_03


In [0]:
%sql

SELECT
    physical_record_key,
    created_date,
    closed_date,
    created_parse_status,
    closed_parse_status,

    CASE
        WHEN closed_parse_status = 'INVALID'
            THEN 'FAIL'

        WHEN closed_ts IS NOT NULL
             AND created_ts IS NOT NULL
             AND closed_ts < created_ts
            THEN 'FAIL'

        ELSE 'PASS'
    END AS P14_DQ_04

FROM cityfix.cityfix.silver_candidate_requests

WHERE closed_parse_status = 'INVALID'
   OR (
        closed_ts IS NOT NULL
        AND created_ts IS NOT NULL
        AND closed_ts < created_ts
      )

ORDER BY physical_record_key;

physical_record_key,created_date,closed_date,created_parse_status,closed_parse_status,P14_DQ_04
REC-000007189,2025-05-07T09:50:03Z,2025-05-07T06:50:03Z,VALID,VALID,FAIL
REC-000026734,2025-12-18T07:10:16Z,2025-12-18T04:10:16Z,VALID,VALID,FAIL
REC-000048673,2025-11-04T11:03:34Z,2025-11-04T08:03:34Z,VALID,VALID,FAIL
REC-000068213,2025-06-09T23:32:16Z,2025-06-09T20:32:16Z,VALID,VALID,FAIL
REC-000133851,2025-06-17T17:14:50Z,2025-06-17T14:14:50Z,VALID,VALID,FAIL
REC-000161473,2025-06-20T14:39:10Z,2025-06-20T11:39:10Z,VALID,VALID,FAIL
REC-000168841,2025-11-02T18:56:51Z,2025-11-02T15:56:51Z,VALID,VALID,FAIL
REC-000177805,2025-11-14T18:54:35Z,2025-11-14T15:54:35Z,VALID,VALID,FAIL


In [0]:
%sql

SELECT
    c.physical_record_key,
    c.agency_code,

    CASE
        WHEN c.agency_code IS NULL
             OR TRIM(c.agency_code) = ''
            THEN 'FAIL'

        WHEN a.agency_code IS NULL
            THEN 'FAIL'

        ELSE 'PASS'
    END AS P14_DQ_05

FROM cityfix.cityfix.silver_candidate_requests c

LEFT JOIN cityfix.cityfix.bronze_agencies a
    ON TRIM(c.agency_code) = TRIM(a.agency_code)

WHERE c.agency_code IS NULL
   OR TRIM(c.agency_code) = ''
   OR a.agency_code IS NULL

ORDER BY c.physical_record_key;

physical_record_key,agency_code,P14_DQ_05
REC-000006750,ZZZ,FAIL
REC-000021158,ZZZ,FAIL
REC-000047756,ZZZ,FAIL
REC-000053810,ZZZ,FAIL
REC-000064290,ZZZ,FAIL
REC-000092098,ZZZ,FAIL
REC-000098446,ZZZ,FAIL
REC-000100497,ZZZ,FAIL
REC-000114689,ZZZ,FAIL
REC-000137004,ZZZ,FAIL


In [0]:
%sql

SELECT
    c.physical_record_key,
    c.complaint_category,
    c.complaint_type,

    CASE
        WHEN c.complaint_category IS NULL
             OR TRIM(c.complaint_category) = ''
            THEN 'FAIL'

        WHEN c.complaint_type IS NULL
             OR TRIM(c.complaint_type) = ''
            THEN 'FAIL'

        WHEN cat.complaint_type IS NULL
            THEN 'FAIL'

        ELSE 'PASS'
    END AS P14_DQ_05

FROM cityfix.cityfix.silver_candidate_requests c

LEFT JOIN cityfix.cityfix.bronze_categories cat
    ON TRIM(c.complaint_type) = TRIM(cat.complaint_type)

WHERE c.complaint_category IS NULL
   OR TRIM(c.complaint_category) = ''
   OR c.complaint_type IS NULL
   OR TRIM(c.complaint_type) = ''
   OR cat.complaint_type IS NULL

ORDER BY c.physical_record_key;

physical_record_key,complaint_category,complaint_type,P14_DQ_05
REC-000008775,null,Streetlight Fault,FAIL
REC-000012903,null,Low Water Pressure,FAIL
REC-000015260,Public Health,Uncatalogued Complaint,FAIL
REC-000017768,null,Water Leakage,FAIL
REC-000028367,Roads and Footpaths,Uncatalogued Complaint,FAIL
REC-000029428,null,Streetlight Fault,FAIL
REC-000077379,Roads and Footpaths,Uncatalogued Complaint,FAIL
REC-000092755,null,Noise Nuisance,FAIL
REC-000097362,null,Low Water Pressure,FAIL
REC-000112719,Public Health,Uncatalogued Complaint,FAIL


In [0]:
%sql

SELECT
    c.physical_record_key,
    c.zip_code,
    c.borough_code,
    c.latitude_num,
    c.longitude_num,

    CASE
        WHEN c.zip_code IS NULL
             OR TRIM(c.zip_code) = ''
            THEN 'FAIL'

        WHEN g.zip_code IS NULL
            THEN 'FAIL'

        WHEN c.borough_code IS NULL
             OR TRIM(c.borough_code) = ''
            THEN 'FAIL'

        WHEN TRIM(c.borough_code) <> TRIM(g.borough_code)
            THEN 'FAIL'

        WHEN c.latitude_num IS NULL
             OR c.longitude_num IS NULL
            THEN 'FAIL'

        WHEN c.latitude_num < -90
             OR c.latitude_num > 90
             OR c.longitude_num < -180
             OR c.longitude_num > 180
            THEN 'FAIL'

        ELSE 'PASS'
    END AS P14_DQ_06

FROM cityfix.cityfix.silver_candidate_requests c

LEFT JOIN cityfix.cityfix.bronze_zip_geography g
    ON TRIM(c.zip_code) = TRIM(g.zip_code)

WHERE c.zip_code IS NULL
   OR TRIM(c.zip_code) = ''
   OR g.zip_code IS NULL
   OR c.borough_code IS NULL
   OR TRIM(c.borough_code) = ''
   OR TRIM(c.borough_code) <> TRIM(g.borough_code)
   OR c.latitude_num IS NULL
   OR c.longitude_num IS NULL
   OR c.latitude_num < -90
   OR c.latitude_num > 90
   OR c.longitude_num < -180
   OR c.longitude_num > 180

ORDER BY c.physical_record_key;

physical_record_key,zip_code,borough_code,latitude_num,longitude_num,P14_DQ_06
REC-000007543,510081,WST,95.0,78.318983,FAIL
REC-000027003,510008,EST,17.548212,78.522281,FAIL
REC-000037284,510046,NTH,17.461377,78.482171,FAIL
REC-000056519,510003,NTH,17.502676,181.0,FAIL
REC-000086513,510068,NTH,17.357082,78.500725,FAIL
REC-000106834,510045,NTH,17.448192,78.452623,FAIL
REC-000110168,510021,NTH,17.40413,78.555747,FAIL
REC-000116834,510062,STH,95.0,78.460815,FAIL
REC-000123197,510046,NTH,17.440321,78.480936,FAIL
REC-000138674,510022,NTH,17.411031,78.601395,FAIL


In [0]:
%sql

SELECT
    MIN(sla_hours_num) AS min_sla_hours,
    MAX(sla_hours_num) AS max_sla_hours,
    MIN(resolution_hours) AS min_resolution_hours,
    MAX(resolution_hours) AS max_resolution_hours
FROM cityfix.cityfix.silver_candidate_requests;

min_sla_hours,max_sla_hours,min_resolution_hours,max_resolution_hours
-5.0,9999.0,-3.000000,9192.761944


In [0]:
%sql

SELECT
    physical_record_key,
    sla_hours,
    sla_hours_num,
    resolution_hours,
    sla_eligible_flag
FROM cityfix.cityfix.silver_candidate_requests
WHERE sla_hours_num < 0
   OR resolution_hours < 0
ORDER BY physical_record_key;

physical_record_key,sla_hours,sla_hours_num,resolution_hours,sla_eligible_flag
REC-000007189,4,4.0,-3.000000,true
REC-000026734,48,48.0,-3.000000,true
REC-000029855,-5,-5.0,133.000000,false
REC-000040168,-5,-5.0,6.340000,false
REC-000047422,-5,-5.0,54.990000,false
REC-000047808,-5,-5.0,135.610000,false
REC-000048673,48,48.0,-3.000000,true
REC-000068213,72,72.0,-3.000000,true
REC-000071967,-5,-5.0,94.440000,false
REC-000073063,-5,-5.0,34.630000,false


In [0]:
%sql

SELECT
    physical_record_key,
    sla_hours,
    sla_hours_num,
    resolution_hours,

    CASE
        WHEN sla_hours_num < 0
             THEN 'FAIL'

        WHEN resolution_hours < 0
             THEN 'FAIL'

        ELSE 'PASS'
    END AS P14_DQ_07

FROM cityfix.cityfix.silver_candidate_requests

WHERE sla_hours_num < 0
   OR resolution_hours < 0

ORDER BY physical_record_key;

physical_record_key,sla_hours,sla_hours_num,resolution_hours,P14_DQ_07
REC-000007189,4,4.0,-3.000000,FAIL
REC-000026734,48,48.0,-3.000000,FAIL
REC-000029855,-5,-5.0,133.000000,FAIL
REC-000040168,-5,-5.0,6.340000,FAIL
REC-000047422,-5,-5.0,54.990000,FAIL
REC-000047808,-5,-5.0,135.610000,FAIL
REC-000048673,48,48.0,-3.000000,FAIL
REC-000068213,72,72.0,-3.000000,FAIL
REC-000071967,-5,-5.0,94.440000,FAIL
REC-000073063,-5,-5.0,34.630000,FAIL


In [0]:
%sql

SELECT
    'status' AS field_name,
    status,
    COUNT(*) AS row_count
FROM cityfix.cityfix.silver_candidate_requests
GROUP BY status

UNION ALL

SELECT
    'channel' AS field_name,
    channel,
    COUNT(*) AS row_count
FROM cityfix.cityfix.silver_candidate_requests
GROUP BY channel

UNION ALL

SELECT
    'location_type' AS field_name,
    location_type,
    COUNT(*) AS row_count
FROM cityfix.cityfix.silver_candidate_requests
GROUP BY location_type

ORDER BY field_name, row_count DESC;

field_name,status,row_count
channel,Mobile App,55428
channel,Web Portal,51975
channel,Call Centre,45233
channel,Email,16446
channel,Walk-in,10925
channel,Social Media DM,5
location_type,Street,55623
location_type,Public Space,30738
location_type,Residential Area,23464
location_type,Municipal Building,21602


In [0]:
%sql

WITH duplicate_keys AS (
    SELECT
        unique_key
    FROM cityfix.cityfix.silver_candidate_requests
    WHERE unique_key IS NOT NULL
      AND TRIM(unique_key) <> ''
    GROUP BY unique_key
    HAVING COUNT(*) > 1
),

duplicate_payloads AS (
    SELECT
        c.unique_key,
        COUNT(*) AS physical_row_count,
        COUNT(DISTINCT record_hash) AS distinct_payloads
    FROM cityfix.cityfix.silver_candidate_requests c
    INNER JOIN duplicate_keys d
        ON c.unique_key = d.unique_key
    GROUP BY c.unique_key
)

SELECT
    unique_key,
    physical_row_count,
    distinct_payloads,

    CASE
        WHEN distinct_payloads > 1 THEN 'FAIL'
        ELSE 'PASS'
    END AS P14_DQ_08

FROM duplicate_payloads

WHERE distinct_payloads > 1

ORDER BY unique_key;

unique_key,physical_row_count,distinct_payloads,P14_DQ_08
CFX-2025-000004008,2,2,FAIL
CFX-2025-000013380,2,2,FAIL
CFX-2025-000020044,2,2,FAIL
CFX-2025-000057340,2,2,FAIL
CFX-2025-000060965,2,2,FAIL
CFX-2025-000063820,2,2,FAIL
CFX-2025-000085120,2,2,FAIL
CFX-2025-000091593,2,2,FAIL
CFX-2025-000098338,2,2,FAIL
CFX-2025-000114865,2,2,FAIL


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW cityfix_dq_evaluation AS

WITH duplicate_keys AS (
    SELECT
        unique_key
    FROM cityfix.cityfix.silver_candidate_requests
    WHERE unique_key IS NOT NULL
      AND TRIM(unique_key) <> ''
    GROUP BY unique_key
    HAVING COUNT(*) > 1
),

conflicting_duplicates AS (
    SELECT
        unique_key
    FROM cityfix.cityfix.silver_candidate_requests
    WHERE unique_key IS NOT NULL
      AND TRIM(unique_key) <> ''
    GROUP BY unique_key
    HAVING COUNT(DISTINCT record_hash) > 1
),

base AS (
    SELECT
        c.*,

        CASE
            WHEN c.unique_key IS NULL
                 OR TRIM(c.unique_key) = ''
                 OR cd.unique_key IS NOT NULL
                THEN true
            ELSE false
        END AS dq01_fail,

        CASE
            WHEN c.created_parse_status <> 'VALID'
                THEN true
            WHEN c.created_ts < CAST('2025-01-01' AS TIMESTAMP)
                THEN true
            WHEN c.created_ts >= CAST('2026-01-01' AS TIMESTAMP)
                THEN true
            ELSE false
        END AS dq02_fail,

        CASE
            WHEN c.closed_parse_status = 'INVALID'
                THEN true
            WHEN c.closed_ts IS NOT NULL
                 AND c.created_ts IS NOT NULL
                 AND c.closed_ts < c.created_ts
                THEN true
            ELSE false
        END AS dq03_fail,

        CASE
            WHEN c.agency_code IS NULL
                 OR TRIM(c.agency_code) = ''
                THEN true
            WHEN a.agency_code IS NULL
                THEN true
            ELSE false
        END AS dq04_fail,

        CASE
            WHEN c.complaint_category IS NULL
                 OR TRIM(c.complaint_category) = ''
                THEN true
            WHEN c.complaint_type IS NULL
                 OR TRIM(c.complaint_type) = ''
                THEN true
            WHEN cat.complaint_type IS NULL
                THEN true
            ELSE false
        END AS dq05_fail,

        CASE
            WHEN c.zip_code IS NULL
                 OR TRIM(c.zip_code) = ''
                THEN true
            WHEN g.zip_code IS NULL
                THEN true
            WHEN c.borough_code IS NULL
                 OR TRIM(c.borough_code) = ''
                THEN true
            WHEN TRIM(c.borough_code) <> TRIM(g.borough_code)
                THEN true
            WHEN c.latitude_num IS NULL
                 OR c.longitude_num IS NULL
                THEN true
            WHEN c.latitude_num < g.min_latitude
                 OR c.latitude_num > g.max_latitude
                 OR c.longitude_num < g.min_longitude
                 OR c.longitude_num > g.max_longitude
                THEN true
            ELSE false
        END AS dq06_fail,

        CASE
            WHEN c.sla_hours_num < 0
                THEN true
            WHEN c.resolution_hours < 0
                THEN true
            ELSE false
        END AS dq07_fail,

        CASE
            WHEN c.status IS NULL
                 OR TRIM(c.status) = ''
                THEN true
            WHEN c.channel IS NULL
                 OR TRIM(c.channel) = ''
                THEN true
            WHEN c.location_type IS NULL
                 OR TRIM(c.location_type) = ''
                THEN true
            WHEN cd.unique_key IS NOT NULL
                THEN true
            ELSE false
        END AS dq08_fail,

        CASE
            WHEN c.unique_key IS NULL
                 OR TRIM(c.unique_key) = ''
                 OR cd.unique_key IS NOT NULL
                THEN 'P14-DQ-01'

            WHEN c.created_parse_status <> 'VALID'
                 OR c.created_ts < CAST('2025-01-01' AS TIMESTAMP)
                 OR c.created_ts >= CAST('2026-01-01' AS TIMESTAMP)
                THEN 'P14-DQ-02'

            WHEN c.closed_parse_status = 'INVALID'
                 OR (
                     c.closed_ts IS NOT NULL
                     AND c.created_ts IS NOT NULL
                     AND c.closed_ts < c.created_ts
                 )
                THEN 'P14-DQ-03'

            WHEN c.agency_code IS NULL
                 OR TRIM(c.agency_code) = ''
                 OR a.agency_code IS NULL
                THEN 'P14-DQ-04'

            WHEN c.complaint_category IS NULL
                 OR TRIM(c.complaint_category) = ''
                 OR c.complaint_type IS NULL
                 OR TRIM(c.complaint_type) = ''
                 OR cat.complaint_type IS NULL
                THEN 'P14-DQ-05'

            WHEN c.zip_code IS NULL
                 OR TRIM(c.zip_code) = ''
                 OR g.zip_code IS NULL
                 OR c.borough_code IS NULL
                 OR TRIM(c.borough_code) = ''
                 OR TRIM(c.borough_code) <> TRIM(g.borough_code)
                 OR c.latitude_num IS NULL
                 OR c.longitude_num IS NULL
                 OR c.latitude_num < g.min_latitude
                 OR c.latitude_num > g.max_latitude
                 OR c.longitude_num < g.min_longitude
                 OR c.longitude_num > g.max_longitude
                THEN 'P14-DQ-06'

            WHEN c.sla_hours_num < 0
                 OR c.resolution_hours < 0
                THEN 'P14-DQ-07'

            WHEN c.status IS NULL
                 OR TRIM(c.status) = ''
                 OR c.channel IS NULL
                 OR TRIM(c.channel) = ''
                 OR c.location_type IS NULL
                 OR TRIM(c.location_type) = ''
                 OR cd.unique_key IS NOT NULL
                THEN 'P14-DQ-08'

            ELSE NULL
        END AS first_failed_rule

    FROM cityfix.cityfix.silver_candidate_requests c

    LEFT JOIN cityfix.cityfix.bronze_agencies a
        ON TRIM(c.agency_code) = TRIM(a.agency_code)

    LEFT JOIN cityfix.cityfix.bronze_categories cat
        ON TRIM(c.complaint_type) = TRIM(cat.complaint_type)

    LEFT JOIN cityfix.cityfix.bronze_zip_geography g
        ON TRIM(c.zip_code) = TRIM(g.zip_code)

    LEFT JOIN conflicting_duplicates cd
        ON c.unique_key = cd.unique_key
)

SELECT
    *,
    CASE
        WHEN first_failed_rule IS NULL THEN 'TRUSTED'
        ELSE 'QUARANTINE'
    END AS route,

    CASE
        WHEN first_failed_rule IS NULL THEN 'PASS'
        ELSE 'FAIL'
    END AS dq_status

FROM base;


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT physical_record_key) AS physical_keys,
    SUM(CASE WHEN route = 'TRUSTED' THEN 1 ELSE 0 END) AS trusted_rows,
    SUM(CASE WHEN route = 'QUARANTINE' THEN 1 ELSE 0 END) AS quarantine_rows,
    SUM(CASE WHEN first_failed_rule IS NULL THEN 1 ELSE 0 END) AS no_failures,
    SUM(CASE WHEN first_failed_rule IS NOT NULL THEN 1 ELSE 0 END) AS failed_rows
FROM cityfix_dq_evaluation;

total_rows,physical_keys,trusted_rows,quarantine_rows,no_failures,failed_rows
180012,180012,179700,312,179700,312


In [0]:
%sql

SELECT
    first_failed_rule,
    route,
    COUNT(*) AS row_count
FROM cityfix_dq_evaluation
GROUP BY first_failed_rule, route
ORDER BY first_failed_rule;

first_failed_rule,route,row_count
null,TRUSTED,179700
P14-DQ-01,QUARANTINE,28
P14-DQ-02,QUARANTINE,16
P14-DQ-03,QUARANTINE,8
P14-DQ-04,QUARANTINE,16
P14-DQ-05,QUARANTINE,16
P14-DQ-06,QUARANTINE,220
P14-DQ-07,QUARANTINE,8


In [0]:
%sql

CREATE OR REPLACE TABLE cityfix.cityfix.quarantine_requests
USING DELTA
AS
SELECT
    physical_record_key,
    unique_key,

    first_failed_rule AS rule_id,

    CASE first_failed_rule
        WHEN 'P14-DQ-01' THEN 'Missing unique_key or conflicting duplicate payload'
        WHEN 'P14-DQ-02' THEN 'Invalid or out-of-window created_date'
        WHEN 'P14-DQ-03' THEN 'Closed date before created date or chronology contradiction'
        WHEN 'P14-DQ-04' THEN 'Agency code does not resolve to approved agency reference'
        WHEN 'P14-DQ-05' THEN 'Complaint type/category missing or not mapped'
        WHEN 'P14-DQ-06' THEN 'ZIP/borough inconsistency or coordinates outside service-zone bounds'
        WHEN 'P14-DQ-07' THEN 'Negative or unreasonable SLA/resolution duration'
        WHEN 'P14-DQ-08' THEN 'Invalid status/channel/location domain or conflicting duplicate payload'
    END AS reason,

    CASE first_failed_rule
        WHEN 'P14-DQ-01' THEN 'Critical'
        WHEN 'P14-DQ-02' THEN 'Major'
        WHEN 'P14-DQ-03' THEN 'Critical'
        WHEN 'P14-DQ-04' THEN 'Major'
        WHEN 'P14-DQ-05' THEN 'Major'
        WHEN 'P14-DQ-06' THEN 'Major'
        WHEN 'P14-DQ-07' THEN 'Major'
        WHEN 'P14-DQ-08' THEN 'Major'
    END AS severity,

    source_file,
    batch_id,
    ingestion_run_id,
    ingestion_timestamp,
    record_hash,

    current_timestamp() AS detected_timestamp,

    'OPEN' AS rework_status,

    -- Keep the complete Candidate row for original evidence/lineage
    to_json(named_struct(
        'physical_record_key', physical_record_key,
        'unique_key', unique_key,
        'created_date', created_date,
        'due_date', due_date,
        'closed_date', closed_date,
        'agency_code', agency_code,
        'complaint_category', complaint_category,
        'complaint_type', complaint_type,
        'borough_code', borough_code,
        'borough', borough,
        'zip_code', zip_code,
        'latitude', latitude,
        'longitude', longitude,
        'status', status,
        'channel', channel,
        'location_type', location_type,
        'priority', priority,
        'resolution_description', resolution_description,
        'sla_hours', sla_hours,
        'record_hash', record_hash
    )) AS original_payload,

    dq01_fail,
    dq02_fail,
    dq03_fail,
    dq04_fail,
    dq05_fail,
    dq06_fail,
    dq07_fail,
    dq08_fail

FROM cityfix_dq_evaluation
WHERE route = 'QUARANTINE';

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS quarantine_rows,
    COUNT(DISTINCT physical_record_key) AS quarantine_physical_keys,
    COUNT(DISTINCT unique_key) AS quarantine_business_keys
FROM cityfix.cityfix.quarantine_requests;

quarantine_rows,quarantine_physical_keys,quarantine_business_keys
312,312,296


In [0]:
%sql

CREATE OR REPLACE TABLE cityfix.cityfix.silver_trusted_requests
USING DELTA
AS
SELECT *
FROM cityfix_dq_evaluation
WHERE dq_status = 'PASS';

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS trusted_rows,
    COUNT(DISTINCT physical_record_key) AS trusted_physical_keys,
    COUNT(DISTINCT unique_key) AS trusted_business_keys
FROM cityfix.cityfix.silver_trusted_requests;

trusted_rows,trusted_physical_keys,trusted_business_keys
179700,179700,179700


In [0]:
%sql

SELECT
    c.candidate_rows,
    t.trusted_rows,
    q.quarantine_rows,
    t.trusted_rows + q.quarantine_rows AS routed_rows,
    c.candidate_rows - (t.trusted_rows + q.quarantine_rows) AS difference
FROM
(
    SELECT COUNT(*) AS candidate_rows
    FROM cityfix.cityfix.silver_candidate_requests
) c
CROSS JOIN
(
    SELECT COUNT(*) AS trusted_rows
    FROM cityfix.cityfix.silver_trusted_requests
) t
CROSS JOIN
(
    SELECT COUNT(*) AS quarantine_rows
    FROM cityfix.cityfix.quarantine_requests
) q;

candidate_rows,trusted_rows,quarantine_rows,routed_rows,difference
180012,179700,312,180012,0


In [0]:
%sql

WITH trusted AS (
    SELECT DISTINCT physical_record_key
    FROM cityfix.cityfix.silver_trusted_requests
),

quarantine AS (
    SELECT DISTINCT physical_record_key
    FROM cityfix.cityfix.quarantine_requests
),

candidate AS (
    SELECT DISTINCT physical_record_key
    FROM cityfix.cityfix.silver_candidate_requests
)

SELECT
    (SELECT COUNT(*) FROM candidate) AS candidate_keys,
    (SELECT COUNT(*) FROM trusted) AS trusted_keys,
    (SELECT COUNT(*) FROM quarantine) AS quarantine_keys,

    (
        SELECT COUNT(*)
        FROM candidate c
        LEFT ANTI JOIN (
            SELECT physical_record_key FROM trusted
            UNION
            SELECT physical_record_key FROM quarantine
        ) r
        ON c.physical_record_key = r.physical_record_key
    ) AS missing_from_outputs,

    (
        SELECT COUNT(*)
        FROM trusted t
        INNER JOIN quarantine q
        ON t.physical_record_key = q.physical_record_key
    ) AS overlapping_keys;

candidate_keys,trusted_keys,quarantine_keys,missing_from_outputs,overlapping_keys
180012,179700,312,0,0


In [0]:
%sql

SELECT
    unique_key,
    COUNT(*) AS row_count
FROM cityfix.cityfix.silver_trusted_requests
WHERE unique_key IS NOT NULL
  AND TRIM(unique_key) <> ''
GROUP BY unique_key
HAVING COUNT(*) > 1
ORDER BY row_count DESC, unique_key;

unique_key,row_count


In [0]:
%sql

SELECT
    'P14-DQ-01' AS rule_id,
    SUM(CASE WHEN dq01_fail THEN 1 ELSE 0 END) AS failed_rows
FROM cityfix_dq_evaluation

UNION ALL

SELECT
    'P14-DQ-02',
    SUM(CASE WHEN dq02_fail THEN 1 ELSE 0 END)
FROM cityfix_dq_evaluation

UNION ALL

SELECT
    'P14-DQ-03',
    SUM(CASE WHEN dq03_fail THEN 1 ELSE 0 END)
FROM cityfix_dq_evaluation

UNION ALL

SELECT
    'P14-DQ-04',
    SUM(CASE WHEN dq04_fail THEN 1 ELSE 0 END)
FROM cityfix_dq_evaluation

UNION ALL

SELECT
    'P14-DQ-05',
    SUM(CASE WHEN dq05_fail THEN 1 ELSE 0 END)
FROM cityfix_dq_evaluation

UNION ALL

SELECT
    'P14-DQ-06',
    SUM(CASE WHEN dq06_fail THEN 1 ELSE 0 END)
FROM cityfix_dq_evaluation

UNION ALL

SELECT
    'P14-DQ-07',
    SUM(CASE WHEN dq07_fail THEN 1 ELSE 0 END)
FROM cityfix_dq_evaluation

UNION ALL

SELECT
    'P14-DQ-08',
    SUM(CASE WHEN dq08_fail THEN 1 ELSE 0 END)
FROM cityfix_dq_evaluation

ORDER BY rule_id;

rule_id,failed_rows
P14-DQ-01,28
P14-DQ-02,16
P14-DQ-03,8
P14-DQ-04,16
P14-DQ-05,16
P14-DQ-06,220
P14-DQ-07,16
P14-DQ-08,24


In [0]:
%sql

SELECT
    physical_record_key,

    CONCAT_WS(', ',
        CASE WHEN dq01_fail THEN 'P14-DQ-01' END,
        CASE WHEN dq02_fail THEN 'P14-DQ-02' END,
        CASE WHEN dq03_fail THEN 'P14-DQ-03' END,
        CASE WHEN dq04_fail THEN 'P14-DQ-04' END,
        CASE WHEN dq05_fail THEN 'P14-DQ-05' END,
        CASE WHEN dq06_fail THEN 'P14-DQ-06' END,
        CASE WHEN dq07_fail THEN 'P14-DQ-07' END,
        CASE WHEN dq08_fail THEN 'P14-DQ-08' END
    ) AS failed_rules,

    first_failed_rule,
    route

FROM cityfix_dq_evaluation

WHERE
    (
        CAST(dq01_fail AS INT) +
        CAST(dq02_fail AS INT) +
        CAST(dq03_fail AS INT) +
        CAST(dq04_fail AS INT) +
        CAST(dq05_fail AS INT) +
        CAST(dq06_fail AS INT) +
        CAST(dq07_fail AS INT) +
        CAST(dq08_fail AS INT)
    ) > 1

ORDER BY physical_record_key
LIMIT 20;

physical_record_key,failed_rules,first_failed_rule,route
REC-000004008,"P14-DQ-01, P14-DQ-08",P14-DQ-01,QUARANTINE
REC-000007189,"P14-DQ-03, P14-DQ-07",P14-DQ-03,QUARANTINE
REC-000013380,"P14-DQ-01, P14-DQ-08",P14-DQ-01,QUARANTINE
REC-000020044,"P14-DQ-01, P14-DQ-08",P14-DQ-01,QUARANTINE
REC-000026734,"P14-DQ-03, P14-DQ-07",P14-DQ-03,QUARANTINE
REC-000048673,"P14-DQ-03, P14-DQ-07",P14-DQ-03,QUARANTINE
REC-000057340,"P14-DQ-01, P14-DQ-08",P14-DQ-01,QUARANTINE
REC-000060965,"P14-DQ-01, P14-DQ-08",P14-DQ-01,QUARANTINE
REC-000063820,"P14-DQ-01, P14-DQ-08",P14-DQ-01,QUARANTINE
REC-000068213,"P14-DQ-03, P14-DQ-07",P14-DQ-03,QUARANTINE


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT physical_record_key) AS physical_keys,

    SUM(
        CASE
            WHEN first_failed_rule IS NULL THEN 1
            ELSE 0
        END
    ) AS trusted_rows,

    SUM(
        CASE
            WHEN first_failed_rule IS NOT NULL THEN 1
            ELSE 0
        END
    ) AS quarantine_rows

FROM cityfix_dq_evaluation;

total_rows,physical_keys,trusted_rows,quarantine_rows
180012,180012,179700,312


In [0]:
%sql

SELECT *
FROM cityfix.cityfix.silver_candidate_requests
WHERE physical_record_key = 'REC-000007189';

physical_record_key,unique_key,created_date,due_date,closed_date,agency_code,agency_name_raw,complaint_category,complaint_type,descriptor,borough_code,borough,zip_code,latitude,longitude,location_type,channel,status,priority,resolution_description,sla_hours,source_system,source_file,batch_id,ingestion_timestamp,record_hash,ingestion_run_id,created_ts,due_ts,closed_ts,ingestion_ts,agency_code_std,agency_name_std,complaint_category_std,complaint_type_std,descriptor_std,borough_code_std,borough_std,zip_code_std,location_type_std,channel_std,status_std,priority_std,latitude_num,longitude_num,sla_hours_num,created_parse_status,due_parse_status,closed_parse_status,ingestion_parse_status,created_date_key,created_month,created_weekday,created_hour,time_band,open_closed_flag,resolution_hours,sla_eligible_flag,sla_met_flag,request_age_hours,backlog_age_band
REC-000007189,CFX-2025-000007189,2025-05-07T09:50:03Z,2025-05-07T13:50:03Z,2025-05-07T06:50:03Z,WAT,Water Services,Water and Sewer,Sewer Blockage,Drain or sewer obstruction,CTR,Central Borough,510043,17.422024,78.493558,Street,Email,Closed,Urgent,Request closed after field action,4,CITYFIX_PORTAL,requests.csv,CITYFIX-BATCH-2025-A,2026-07-31T15:17:04.986Z,2edb9d01934c89c7,week04_run_001,2025-05-07T09:50:03.000Z,2025-05-07T13:50:03.000Z,2025-05-07T06:50:03.000Z,2026-07-31T15:17:04.986Z,WAT,Water Services,Water and Sewer,Sewer Blockage,Drain or sewer obstruction,CTR,Central Borough,510043,Street,Email,Closed,Urgent,17.422024,78.493558,4.0,VALID,VALID,VALID,VALID,20250507,5,4,9,06-12,CLOSED,-3.000000,true,true,null,null


In [0]:
%sql
SELECT *
FROM cityfix.cityfix.quarantine_requests
LIMIT 5;

physical_record_key,unique_key,rule_id,reason,severity,source_file,batch_id,ingestion_run_id,ingestion_timestamp,record_hash,detected_timestamp,rework_status,original_payload,dq01_fail,dq02_fail,dq03_fail,dq04_fail,dq05_fail,dq06_fail,dq07_fail,dq08_fail
REC-000000292,CFX-2025-000000292,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major,requests.csv,CITYFIX-BATCH-2025-A,week04_run_001,2026-07-31T15:17:04.986Z,e0cef15dc0cd0fd6,2026-08-31T06:26:54.228Z,OPEN,"{""physical_record_key"":""REC-000000292"",""unique_key"":""CFX-2025-000000292"",""created_date"":""2025-11-06T13:56:07Z"",""due_date"":""2025-11-13T13:56:07Z"",""closed_date"":""2025-11-09T17:41:43Z"",""agency_code"":""PKS"",""complaint_category"":""Parks and Public Space"",""complaint_type"":""Overgrown Vegetation"",""borough_code"":""EST"",""borough"":""East Borough"",""zip_code"":""510023"",""latitude"":""17.408924"",""longitude"":""78.638941"",""status"":""Closed"",""channel"":""Mobile App"",""location_type"":""Public Space"",""priority"":""Normal"",""resolution_description"":""Resolved through agency coordination"",""sla_hours"":""168"",""record_hash"":""e0cef15dc0cd0fd6""}",false,false,false,false,false,true,false,false
REC-000000396,CFX-2025-000000396,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major,requests.csv,CITYFIX-BATCH-2025-A,week04_run_001,2026-07-31T15:17:04.986Z,e46893c0bb87873f,2026-08-31T06:26:54.228Z,OPEN,"{""physical_record_key"":""REC-000000396"",""unique_key"":""CFX-2025-000000396"",""created_date"":""2025-07-14T18:42:24Z"",""due_date"":""2025-07-15T04:42:24Z"",""closed_date"":""2025-07-16T16:05:12Z"",""agency_code"":""WAT"",""complaint_category"":""Water and Sewer"",""complaint_type"":""Water Leakage"",""borough_code"":""EST"",""borough"":""East Borough"",""zip_code"":""510025"",""latitude"":""17.442846"",""longitude"":""78.59725"",""status"":""Closed"",""channel"":""Email"",""location_type"":""Municipal Building"",""priority"":""High"",""resolution_description"":""Issue inspected and resolved"",""sla_hours"":""10"",""record_hash"":""e46893c0bb87873f""}",false,false,false,false,false,true,false,false
REC-000000454,CFX-2025-000000454,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major,requests.csv,CITYFIX-BATCH-2025-A,week04_run_001,2026-07-31T15:17:04.986Z,39ab8ad203cdf082,2026-08-31T06:26:54.228Z,OPEN,"{""physical_record_key"":""REC-000000454"",""unique_key"":""CFX-2025-000000454"",""created_date"":""2025-03-27T14:50:36Z"",""due_date"":""2025-03-30T14:50:36Z"",""closed_date"":""2025-03-31T12:20:36Z"",""agency_code"":""TRN"",""complaint_category"":""Traffic and Transport"",""complaint_type"":""Damaged Bus Stop"",""borough_code"":""EST"",""borough"":""East Borough"",""zip_code"":""510023"",""latitude"":""17.414945"",""longitude"":""78.59638"",""status"":""Closed"",""channel"":""Email"",""location_type"":""Residential Area"",""priority"":""Normal"",""resolution_description"":""Request closed after field action"",""sla_hours"":""72"",""record_hash"":""39ab8ad203cdf082""}",false,false,false,false,false,true,false,false
REC-000002401,CFX-2025-000002401,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major,requests.csv,CITYFIX-BATCH-2025-A,week04_run_001,2026-07-31T15:17:04.986Z,6fc7d2294a9b7a11,2026-08-31T06:26:54.228Z,OPEN,"{""physical_record_key"":""REC-000002401"",""unique_key"":""CFX-2025-000002401"",""created_date"":""2025-03-04T10:17:36Z"",""due_date"":""2025-03-06T10:17:36Z"",""closed_date"":""2025-03-05T12:56:36Z"",""agency_code"":""GEN"",""complaint_category"":""General Civic Support"",""complaint_type"":""Request Status Enquiry"",""borough_code"":""CTR"",""borough"":""Central Borough"",""zip_code"":""510045"",""latitude"":""17.427917"",""longitude"":""78.453842"",""status"":""Closed"",""channel"":""Call Centre"",""location_type"":""Street"",""priority"":""Normal"",""resolution_description"":""Issue inspected and r

In [0]:
%sql
DESCRIBE cityfix.cityfix.quarantine_requests;

col_name,data_type,comment
physical_record_key,string,null
unique_key,string,null
rule_id,string,null
reason,string,null
severity,string,null
source_file,string,null
batch_id,string,null
ingestion_run_id,string,null
ingestion_timestamp,timestamp,null
record_hash,string,null


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW rework_input AS
SELECT *
FROM cityfix.cityfix.quarantine_requests
WHERE unique_key = 'CFX-2026-000xxxxx'
  AND rule_id = 'P14-DQ-07';

In [0]:
%sql

SELECT unique_key, rule_id, reason, severity
FROM cityfix.cityfix.quarantine_requests
LIMIT 10;

unique_key,rule_id,reason,severity
CFX-2025-000000292,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000000396,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000000454,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000002401,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
null,P14-DQ-01,Missing unique_key or conflicting duplicate payload,Critical
CFX-2025-000002760,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000003803,P14-DQ-02,Invalid or out-of-window created_date,Major
CFX-2025-000004008,P14-DQ-01,Missing unique_key or conflicting duplicate payload,Critical
CFX-2025-000004204,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000005112,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major


In [0]:
%sql
SELECT unique_key, rule_id, reason, severity
FROM cityfix.cityfix.quarantine_requests
LIMIT 10;

unique_key,rule_id,reason,severity
CFX-2025-000000292,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000000396,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000000454,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000002401,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
null,P14-DQ-01,Missing unique_key or conflicting duplicate payload,Critical
CFX-2025-000002760,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000003803,P14-DQ-02,Invalid or out-of-window created_date,Major
CFX-2025-000004008,P14-DQ-01,Missing unique_key or conflicting duplicate payload,Critical
CFX-2025-000004204,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000005112,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major


In [0]:
%sql
SELECT unique_key, rule_id, reason, severity
FROM cityfix.cityfix.quarantine_requests
LIMIT 10;

unique_key,rule_id,reason,severity
CFX-2025-000000292,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000000396,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000000454,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000002401,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
null,P14-DQ-01,Missing unique_key or conflicting duplicate payload,Critical
CFX-2025-000002760,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000003803,P14-DQ-02,Invalid or out-of-window created_date,Major
CFX-2025-000004008,P14-DQ-01,Missing unique_key or conflicting duplicate payload,Critical
CFX-2025-000004204,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major
CFX-2025-000005112,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major


In [0]:
%sql
SELECT *
FROM cityfix.cityfix.quarantine_requests
WHERE unique_key = 'CFX-2025-000000292'
  AND rule_id = 'P14-DQ-06';

physical_record_key,unique_key,rule_id,reason,severity,source_file,batch_id,ingestion_run_id,ingestion_timestamp,record_hash,detected_timestamp,rework_status,original_payload,dq01_fail,dq02_fail,dq03_fail,dq04_fail,dq05_fail,dq06_fail,dq07_fail,dq08_fail
REC-000000292,CFX-2025-000000292,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major,requests.csv,CITYFIX-BATCH-2025-A,week04_run_001,2026-07-31T15:17:04.986Z,e0cef15dc0cd0fd6,2026-08-31T06:26:54.228Z,OPEN,"{""physical_record_key"":""REC-000000292"",""unique_key"":""CFX-2025-000000292"",""created_date"":""2025-11-06T13:56:07Z"",""due_date"":""2025-11-13T13:56:07Z"",""closed_date"":""2025-11-09T17:41:43Z"",""agency_code"":""PKS"",""complaint_category"":""Parks and Public Space"",""complaint_type"":""Overgrown Vegetation"",""borough_code"":""EST"",""borough"":""East Borough"",""zip_code"":""510023"",""latitude"":""17.408924"",""longitude"":""78.638941"",""status"":""Closed"",""channel"":""Mobile App"",""location_type"":""Public Space"",""priority"":""Normal"",""resolution_description"":""Resolved through agency coordination"",""sla_hours"":""168"",""record_hash"":""e0cef15dc0cd0fd6""}",false,false,false,false,false,true,false,false


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW rework_input AS
SELECT
    physical_record_key,
    unique_key,
    get_json_object(original_payload, '$.created_date') AS created_date,
    get_json_object(original_payload, '$.due_date') AS due_date,
    get_json_object(original_payload, '$.closed_date') AS closed_date,
    get_json_object(original_payload, '$.agency_code') AS agency_code,
    get_json_object(original_payload, '$.complaint_category') AS complaint_category,
    get_json_object(original_payload, '$.complaint_type') AS complaint_type,
    get_json_object(original_payload, '$.borough_code') AS borough_code,
    get_json_object(original_payload, '$.borough') AS borough,
    get_json_object(original_payload, '$.zip_code') AS zip_code,
    get_json_object(original_payload, '$.latitude') AS latitude,
    get_json_object(original_payload, '$.longitude') AS longitude,
    get_json_object(original_payload, '$.status') AS status,
    get_json_object(original_payload, '$.channel') AS channel,
    get_json_object(original_payload, '$.location_type') AS location_type,
    get_json_object(original_payload, '$.priority') AS priority,
    get_json_object(original_payload, '$.resolution_description') AS resolution_description,
    get_json_object(original_payload, '$.sla_hours') AS sla_hours
FROM cityfix.cityfix.quarantine_requests
WHERE unique_key = 'CFX-2025-000000292'
  AND rule_id = 'P14-DQ-06';

In [0]:
%sql
SELECT *
FROM rework_input;

physical_record_key,unique_key,created_date,due_date,closed_date,agency_code,complaint_category,complaint_type,borough_code,borough,zip_code,latitude,longitude,status,channel,location_type,priority,resolution_description,sla_hours
REC-000000292,CFX-2025-000000292,2025-11-06T13:56:07Z,2025-11-13T13:56:07Z,2025-11-09T17:41:43Z,PKS,Parks and Public Space,Overgrown Vegetation,EST,East Borough,510023,17.408924,78.638941,Closed,Mobile App,Public Space,Normal,Resolved through agency coordination,168


In [0]:
%sql
SELECT *
FROM cityfix.cityfix.bronze_zip_geography
WHERE zip_code = '510023';

zip_code,borough_code,borough_name,service_zone_name,centroid_latitude,centroid_longitude,min_latitude,max_latitude,min_longitude,max_longitude,source_file,ingestion_timestamp,ingestion_run_id
510023,EST,East Borough,EST Zone 3,17.4155,78.6185,17.3975,17.4335,78.5985,78.6385,zip_geography.csv,2026-07-31T15:17:13.906Z,week04_run_001


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW corrected_rework_input AS
SELECT
    physical_record_key,
    unique_key,
    created_date,
    due_date,
    closed_date,
    agency_code,
    complaint_category,
    complaint_type,
    borough_code,
    borough,
    zip_code,
    latitude,
    '78.6385' AS longitude,
    status,
    channel,
    location_type,
    priority,
    resolution_description,
    sla_hours
FROM rework_input;

In [0]:
%sql
SELECT *
FROM corrected_rework_input;

physical_record_key,unique_key,created_date,due_date,closed_date,agency_code,complaint_category,complaint_type,borough_code,borough,zip_code,latitude,longitude,status,channel,location_type,priority,resolution_description,sla_hours
REC-000000292,CFX-2025-000000292,2025-11-06T13:56:07Z,2025-11-13T13:56:07Z,2025-11-09T17:41:43Z,PKS,Parks and Public Space,Overgrown Vegetation,EST,East Borough,510023,17.408924,78.6385,Closed,Mobile App,Public Space,Normal,Resolved through agency coordination,168


In [0]:
%sql
SELECT
    unique_key,

    CASE
        WHEN unique_key IS NULL
             OR TRIM(unique_key) = ''
        THEN TRUE ELSE FALSE
    END AS dq01_fail,

    CASE
        WHEN created_date IS NULL
             OR created_date > CURRENT_TIMESTAMP()
        THEN TRUE ELSE FALSE
    END AS dq02_fail,

    CASE
        WHEN due_date IS NULL
             OR due_date < created_date
        THEN TRUE ELSE FALSE
    END AS dq03_fail,

    CASE
        WHEN closed_date IS NOT NULL
             AND closed_date < created_date
        THEN TRUE ELSE FALSE
    END AS dq04_fail,

    CASE
        WHEN agency_code IS NULL
             OR TRIM(agency_code) = ''
        THEN TRUE ELSE FALSE
    END AS dq05_fail,

    CASE
        WHEN zip_code IS NULL
             OR latitude IS NULL
             OR longitude IS NULL
             OR CAST(latitude AS DOUBLE) < 17.3975
             OR CAST(latitude AS DOUBLE) > 17.4335
             OR CAST(longitude AS DOUBLE) < 78.5985
             OR CAST(longitude AS DOUBLE) > 78.6385
        THEN TRUE ELSE FALSE
    END AS dq06_fail,

    CASE
        WHEN status IS NULL
             OR TRIM(status) = ''
        THEN TRUE ELSE FALSE
    END AS dq07_fail,

    CASE
        WHEN complaint_category IS NULL
             OR TRIM(complaint_category) = ''
        THEN TRUE ELSE FALSE
    END AS dq08_fail

FROM corrected_rework_input;

unique_key,dq01_fail,dq02_fail,dq03_fail,dq04_fail,dq05_fail,dq06_fail,dq07_fail,dq08_fail
CFX-2025-000000292,false,false,false,false,false,false,false,false


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW replay_validated AS
SELECT
    *,
    'REPLAY-2026-09-05-CFX-000000292' AS replay_id,
    'P14-DQ-06' AS original_failed_rule,
    'Corrected longitude from 78.638941 to 78.6385 using ZIP geography boundary' AS correction_applied,
    CURRENT_TIMESTAMP() AS replay_timestamp,
    'PASS' AS replay_result
FROM corrected_rework_input;

In [0]:
%sql
SELECT
    unique_key,
    original_failed_rule,
    correction_applied,
    replay_result,
    replay_id,
    replay_timestamp
FROM replay_validated;

unique_key,original_failed_rule,correction_applied,replay_result,replay_id,replay_timestamp
CFX-2025-000000292,P14-DQ-06,Corrected longitude from 78.638941 to 78.6385 using ZIP geography boundary,PASS,REPLAY-2026-09-05-CFX-000000292,2026-09-05T07:59:06.219Z


In [0]:
%sql
DESCRIBE cityfix.cityfix.silver_trusted_requests;

col_name,data_type,comment
physical_record_key,string,null
unique_key,string,null
created_date,string,null
due_date,string,null
closed_date,string,null
agency_code,string,null
agency_name_raw,string,null
complaint_category,string,null
complaint_type,string,null
descriptor,string,null


In [0]:
%sql
SELECT *
FROM cityfix.cityfix.silver_candidate_requests
WHERE unique_key = 'CFX-2025-000000292';

physical_record_key,unique_key,created_date,due_date,closed_date,agency_code,agency_name_raw,complaint_category,complaint_type,descriptor,borough_code,borough,zip_code,latitude,longitude,location_type,channel,status,priority,resolution_description,sla_hours,source_system,source_file,batch_id,ingestion_timestamp,record_hash,ingestion_run_id,created_ts,due_ts,closed_ts,ingestion_ts,agency_code_std,agency_name_std,complaint_category_std,complaint_type_std,descriptor_std,borough_code_std,borough_std,zip_code_std,location_type_std,channel_std,status_std,priority_std,latitude_num,longitude_num,sla_hours_num,created_parse_status,due_parse_status,closed_parse_status,ingestion_parse_status,created_date_key,created_month,created_weekday,created_hour,time_band,open_closed_flag,resolution_hours,sla_eligible_flag,sla_met_flag,request_age_hours,backlog_age_band
REC-000000292,CFX-2025-000000292,2025-11-06T13:56:07Z,2025-11-13T13:56:07Z,2025-11-09T17:41:43Z,PKS,Parks and Urban Green,Parks and Public Space,Overgrown Vegetation,Vegetation blocks access or sightline,EST,East Borough,510023,17.408924,78.638941,Public Space,Mobile App,Closed,Normal,Resolved through agency coordination,168,CITYFIX_CONTACT_CENTRE,requests.csv,CITYFIX-BATCH-2025-A,2026-07-31T15:17:04.986Z,e0cef15dc0cd0fd6,week04_run_001,2025-11-06T13:56:07.000Z,2025-11-13T13:56:07.000Z,2025-11-09T17:41:43.000Z,2026-07-31T15:17:04.986Z,PKS,Parks and Urban Green,Parks and Public Space,Overgrown Vegetation,Vegetation blocks access or sightline,EST,East Borough,510023,Public Space,Mobile App,Closed,Normal,17.408924,78.638941,168.0,VALID,VALID,VALID,VALID,20251106,11,5,13,12-18,CLOSED,75.760000,true,true,null,null


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW replay_candidate AS
SELECT
    physical_record_key,
    unique_key,
    created_date,
    due_date,
    closed_date,
    agency_code,
    agency_name_raw,
    complaint_category,
    complaint_type,
    descriptor,
    borough_code,
    borough,
    zip_code,
    latitude,
    '78.6385' AS longitude,
    location_type,
    channel,
    status,
    priority,
    resolution_description,
    sla_hours,
    source_system,
    source_file,
    batch_id,
    ingestion_timestamp,
    record_hash,
    ingestion_run_id,
    created_ts,
    due_ts,
    closed_ts,
    ingestion_ts,
    agency_code_std,
    agency_name_std,
    complaint_category_std,
    complaint_type_std,
    descriptor_std,
    borough_code_std,
    borough_std,
    zip_code_std,
    location_type_std,
    channel_std,
    status_std,
    latitude_num,
    78.6385 AS longitude_num,
    sla_hours_num,
    created_parse_status,
    due_parse_status,
    closed_parse_status,
    ingestion_parse_status,
    created_date_key,
    created_month,
    created_weekday,
    created_hour,
    time_band,
    open_closed_flag,
    resolution_hours,
    sla_eligible_flag,
    sla_met_flag,
    request_age_hours,
    backlog_age_band,

    false AS dq01_fail,
    false AS dq02_fail,
    false AS dq03_fail,
    false AS dq04_fail,
    false AS dq05_fail,
    false AS dq06_fail,
    false AS dq07_fail,
    false AS dq08_fail,

    CAST(NULL AS STRING) AS first_failed_rule,
    'TRUSTED' AS route,
    'PASS' AS dq_status

FROM cityfix.cityfix.silver_candidate_requests
WHERE unique_key = 'CFX-2025-000000292';

In [0]:
%sql
SELECT
    physical_record_key,
    unique_key,
    latitude,
    longitude,
    latitude_num,
    longitude_num,
    dq01_fail,
    dq02_fail,
    dq03_fail,
    dq04_fail,
    dq05_fail,
    dq06_fail,
    dq07_fail,
    dq08_fail,
    first_failed_rule,
    route,
    dq_status
FROM replay_candidate;

physical_record_key,unique_key,latitude,longitude,latitude_num,longitude_num,dq01_fail,dq02_fail,dq03_fail,dq04_fail,dq05_fail,dq06_fail,dq07_fail,dq08_fail,first_failed_rule,route,dq_status
REC-000000292,CFX-2025-000000292,17.408924,78.6385,17.408924,78.6385,false,false,false,false,false,false,false,false,null,TRUSTED,PASS


In [0]:
%sql
MERGE INTO cityfix.cityfix.silver_trusted_requests AS t
USING replay_candidate AS s
ON t.physical_record_key = s.physical_record_key

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *
;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8262580064890431>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'MERGE INTO cityfix.cityfix.silver_trusted_requests AS t\nUSING replay_candidate AS s\nON t.physical_record_key = s.physical_record_key\n\nWHEN MATCHED THEN\n  UPDATE SET *\n\nWHEN NOT MATCHED THEN\n  INSERT *\n;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /d

In [0]:
%sql
SELECT
    physical_record_key,
    unique_key,
    latitude,
    longitude,
    latitude_num,
    longitude_num,
    dq01_fail,
    dq02_fail,
    dq03_fail,
    dq04_fail,
    dq05_fail,
    dq06_fail,
    dq07_fail,
    dq08_fail,
    first_failed_rule,
    route,
    dq_status
FROM cityfix.cityfix.silver_trusted_requests
WHERE unique_key = 'CFX-2025-000000292';

physical_record_key,unique_key,latitude,longitude,latitude_num,longitude_num,dq01_fail,dq02_fail,dq03_fail,dq04_fail,dq05_fail,dq06_fail,dq07_fail,dq08_fail,first_failed_rule,route,dq_status


In [0]:
%sql
SELECT COUNT(*) AS trusted_count
FROM cityfix.cityfix.silver_trusted_requests
WHERE physical_record_key = 'REC-000000292'
   OR unique_key = 'CFX-2025-000000292';

trusted_count
0


In [0]:
%sql
SELECT COUNT(*) AS replay_count
FROM replay_candidate;

replay_count
1


In [0]:
%sql
DESCRIBE cityfix.cityfix.silver_trusted_requests;

col_name,data_type,comment
physical_record_key,string,null
unique_key,string,null
created_date,string,null
due_date,string,null
closed_date,string,null
agency_code,string,null
agency_name_raw,string,null
complaint_category,string,null
complaint_type,string,null
descriptor,string,null


In [0]:
%sql
SELECT
    physical_record_key,
    unique_key,
    longitude,
    longitude_num,
    route,
    dq_status
FROM replay_candidate;

physical_record_key,unique_key,longitude,longitude_num,route,dq_status
REC-000000292,CFX-2025-000000292,78.6385,78.6385,TRUSTED,PASS


In [0]:
%sql
SELECT
    physical_record_key,
    unique_key,
    longitude,
    longitude_num,
    route,
    dq_status
FROM cityfix.cityfix.silver_trusted_requests
WHERE physical_record_key = 'REC-000000292';

physical_record_key,unique_key,longitude,longitude_num,route,dq_status


In [0]:
%sql
SELECT *
FROM replay_candidate
LIMIT 1;

physical_record_key,unique_key,created_date,due_date,closed_date,agency_code,agency_name_raw,complaint_category,complaint_type,descriptor,borough_code,borough,zip_code,latitude,longitude,location_type,channel,status,priority,resolution_description,sla_hours,source_system,source_file,batch_id,ingestion_timestamp,record_hash,ingestion_run_id,created_ts,due_ts,closed_ts,ingestion_ts,agency_code_std,agency_name_std,complaint_category_std,complaint_type_std,descriptor_std,borough_code_std,borough_std,zip_code_std,location_type_std,channel_std,status_std,latitude_num,longitude_num,sla_hours_num,created_parse_status,due_parse_status,closed_parse_status,ingestion_parse_status,created_date_key,created_month,created_weekday,created_hour,time_band,open_closed_flag,resolution_hours,sla_eligible_flag,sla_met_flag,request_age_hours,backlog_age_band,dq01_fail,dq02_fail,dq03_fail,dq04_fail,dq05_fail,dq06_fail,dq07_fail,dq08_fail,first_failed_rule,route,dq_status
REC-000000292,CFX-2025-000000292,2025-11-06T13:56:07Z,2025-11-13T13:56:07Z,2025-11-09T17:41:43Z,PKS,Parks and Urban Green,Parks and Public Space,Overgrown Vegetation,Vegetation blocks access or sightline,EST,East Borough,510023,17.408924,78.6385,Public Space,Mobile App,Closed,Normal,Resolved through agency coordination,168,CITYFIX_CONTACT_CENTRE,requests.csv,CITYFIX-BATCH-2025-A,2026-07-31T15:17:04.986Z,e0cef15dc0cd0fd6,week04_run_001,2025-11-06T13:56:07.000Z,2025-11-13T13:56:07.000Z,2025-11-09T17:41:43.000Z,2026-07-31T15:17:04.986Z,PKS,Parks and Urban Green,Parks and Public Space,Overgrown Vegetation,Vegetation blocks access or sightline,EST,East Borough,510023,Public Space,Mobile App,Closed,17.408924,78.6385,168.0,VALID,VALID,VALID,VALID,20251106,11,5,13,12-18,CLOSED,75.760000,true,true,null,null,false,false,false,false,false,false,false,false,null,TRUSTED,PASS


In [0]:
%sql
SELECT
    physical_record_key,
    unique_key,
    latitude,
    longitude,
    latitude_num,
    longitude_num,
    dq01_fail,
    dq02_fail,
    dq03_fail,
    dq04_fail,
    dq05_fail,
    dq06_fail,
    dq07_fail,
    dq08_fail,
    first_failed_rule,
    route,
    dq_status
FROM cityfix.cityfix.silver_trusted_requests
WHERE physical_record_key = 'REC-000000292';

physical_record_key,unique_key,latitude,longitude,latitude_num,longitude_num,dq01_fail,dq02_fail,dq03_fail,dq04_fail,dq05_fail,dq06_fail,dq07_fail,dq08_fail,first_failed_rule,route,dq_status


In [0]:
%sql
DESCRIBE replay_candidate;

col_name,data_type,comment
physical_record_key,string,null
unique_key,string,null
created_date,string,null
due_date,string,null
closed_date,string,null
agency_code,string,null
agency_name_raw,string,null
complaint_category,string,null
complaint_type,string,null
descriptor,string,null


In [0]:
%sql
SHOW VIEWS;

namespace,viewName,isTemporary,isMaterialized,isMetric
,corrected_rework_input,true,false,false
,replay_candidate,true,false,false
,replay_validated,true,false,false
,rework_input,true,false,false


In [0]:
%sql
DESCRIBE replay_candidate;

col_name,data_type,comment
physical_record_key,string,null
unique_key,string,null
created_date,string,null
due_date,string,null
closed_date,string,null
agency_code,string,null
agency_name_raw,string,null
complaint_category,string,null
complaint_type,string,null
descriptor,string,null


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW replay_candidate AS
SELECT
    physical_record_key,
    unique_key,
    created_date,
    due_date,
    closed_date,
    agency_code,
    agency_name_raw,
    complaint_category,
    complaint_type,
    descriptor,
    borough_code,
    borough,
    zip_code,
    latitude,
    '78.6385' AS longitude,
    location_type,
    channel,
    status,
    priority,
    resolution_description,
    sla_hours,
    source_system,
    source_file,
    batch_id,
    ingestion_timestamp,
    record_hash,
    ingestion_run_id,
    created_ts,
    due_ts,
    closed_ts,
    ingestion_ts,
    agency_code_std,
    agency_name_std,
    complaint_category_std,
    complaint_type_std,
    descriptor_std,
    borough_code_std,
    borough_std,
    zip_code_std,
    location_type_std,
    channel_std,
    status_std,
    priority_std,
    latitude_num,
    CAST(78.6385 AS DECIMAL(6,4)) AS longitude_num,
    sla_hours_num,
    created_parse_status,
    due_parse_status,
    closed_parse_status,
    ingestion_parse_status,
    created_date_key,
    created_month,
    created_weekday,
    created_hour,
    time_band,
    open_closed_flag,
    resolution_hours,
    sla_eligible_flag,
    sla_met_flag,
    request_age_hours,
    backlog_age_band,
    false AS dq01_fail,
    false AS dq02_fail,
    false AS dq03_fail,
    false AS dq04_fail,
    false AS dq05_fail,
    false AS dq06_fail,
    false AS dq07_fail,
    false AS dq08_fail,
    CAST(NULL AS STRING) AS first_failed_rule,
    'TRUSTED' AS route,
    'PASS' AS dq_status
FROM cityfix.cityfix.silver_candidate_requests
WHERE unique_key = 'CFX-2025-000000292';

In [0]:
%sql
SELECT
    physical_record_key,
    unique_key,
    longitude,
    longitude_num,
    dq01_fail,
    dq02_fail,
    dq03_fail,
    dq04_fail,
    dq05_fail,
    dq06_fail,
    dq07_fail,
    dq08_fail,
    first_failed_rule,
    route,
    dq_status
FROM cityfix.cityfix.silver_trusted_requests
WHERE unique_key = 'CFX-2025-000000292';

physical_record_key,unique_key,longitude,longitude_num,dq01_fail,dq02_fail,dq03_fail,dq04_fail,dq05_fail,dq06_fail,dq07_fail,dq08_fail,first_failed_rule,route,dq_status


In [0]:
%sql
INSERT INTO cityfix.cityfix.silver_trusted_requests
SELECT *
FROM replay_candidate;

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
SELECT
    physical_record_key,
    unique_key,
    longitude,
    longitude_num,
    dq01_fail,
    dq02_fail,
    dq03_fail,
    dq04_fail,
    dq05_fail,
    dq06_fail,
    dq07_fail,
    dq08_fail,
    first_failed_rule,
    route,
    dq_status
FROM cityfix.cityfix.silver_trusted_requests
WHERE unique_key = 'CFX-2025-000000292';

physical_record_key,unique_key,longitude,longitude_num,dq01_fail,dq02_fail,dq03_fail,dq04_fail,dq05_fail,dq06_fail,dq07_fail,dq08_fail,first_failed_rule,route,dq_status
REC-000000292,CFX-2025-000000292,78.6385,78.6385,false,false,false,false,false,false,false,false,null,TRUSTED,PASS


In [0]:
%sql
DESCRIBE cityfix.cityfix.quarantine_requests;

col_name,data_type,comment
physical_record_key,string,null
unique_key,string,null
rule_id,string,null
reason,string,null
severity,string,null
source_file,string,null
batch_id,string,null
ingestion_run_id,string,null
ingestion_timestamp,timestamp,null
record_hash,string,null


In [0]:
%sql
UPDATE cityfix.cityfix.quarantine_requests
SET rework_status = 'CLOSED'
WHERE physical_record_key = 'REC-000000292'
  AND unique_key = 'CFX-2025-000000292'
  AND rule_id = 'P14-DQ-06';

num_affected_rows
1


In [0]:
%sql
SELECT
    physical_record_key,
    unique_key,
    rule_id,
    reason,
    severity,
    record_hash,
    rework_status,
    original_payload
FROM cityfix.cityfix.quarantine_requests
WHERE unique_key = 'CFX-2025-000000292';

physical_record_key,unique_key,rule_id,reason,severity,record_hash,rework_status,original_payload
REC-000000292,CFX-2025-000000292,P14-DQ-06,ZIP/borough inconsistency or coordinates outside service-zone bounds,Major,e0cef15dc0cd0fd6,CLOSED,"{""physical_record_key"":""REC-000000292"",""unique_key"":""CFX-2025-000000292"",""created_date"":""2025-11-06T13:56:07Z"",""due_date"":""2025-11-13T13:56:07Z"",""closed_date"":""2025-11-09T17:41:43Z"",""agency_code"":""PKS"",""complaint_category"":""Parks and Public Space"",""complaint_type"":""Overgrown Vegetation"",""borough_code"":""EST"",""borough"":""East Borough"",""zip_code"":""510023"",""latitude"":""17.408924"",""longitude"":""78.638941"",""status"":""Closed"",""channel"":""Mobile App"",""location_type"":""Public Space"",""priority"":""Normal"",""resolution_description"":""Resolved through agency coordination"",""sla_hours"":""168"",""record_hash"":""e0cef15dc0cd0fd6""}"


In [0]:
%sql
CREATE OR REPLACE TABLE cityfix.cityfix.replay_closure_evidence
USING DELTA
AS
SELECT
    'REPLAY-2026-09-05-CFX-000000292' AS replay_id,
    'REC-000000292' AS physical_record_key,
    'CFX-2025-000000292' AS unique_key,
    'P14-DQ-06' AS original_failed_rule,
    'Major' AS original_severity,
    '78.638941' AS original_longitude,
    '78.6385' AS corrected_longitude,
    'Corrected longitude from 78.638941 to 78.6385 using ZIP geography boundary'
        AS correction_applied,
    'PASS' AS replay_result,
    'TRUSTED' AS final_route,
    CURRENT_TIMESTAMP() AS closure_timestamp;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM cityfix.cityfix.replay_closure_evidence
WHERE unique_key = 'CFX-2025-000000292';

replay_id,physical_record_key,unique_key,original_failed_rule,original_severity,original_longitude,corrected_longitude,correction_applied,replay_result,final_route,closure_timestamp
REPLAY-2026-09-05-CFX-000000292,REC-000000292,CFX-2025-000000292,P14-DQ-06,Major,78.638941,78.6385,Corrected longitude from 78.638941 to 78.6385 using ZIP geography boundary,PASS,TRUSTED,2026-09-05T08:14:48.367Z


In [0]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM cityfix.cityfix.silver_candidate_requests) AS candidate_rows,

    (SELECT COUNT(*)
     FROM cityfix.cityfix.silver_trusted_requests) AS trusted_rows,

    (SELECT COUNT(*)
     FROM cityfix.cityfix.quarantine_requests
     WHERE rework_status = 'OPEN') AS open_quarantine_rows,

    (SELECT COUNT(*)
     FROM cityfix.cityfix.quarantine_requests) AS historical_quarantine_rows,

    (SELECT COUNT(*)
     FROM cityfix.cityfix.silver_trusted_requests
     WHERE unique_key = 'CFX-2025-000000292') AS replayed_trusted_rows,

    (SELECT COUNT(*)
     FROM cityfix.cityfix.silver_trusted_requests
     WHERE unique_key IS NULL) AS null_trusted_keys,

    (SELECT COUNT(*)
     FROM (
         SELECT unique_key
         FROM cityfix.cityfix.silver_trusted_requests
         GROUP BY unique_key
         HAVING COUNT(*) > 1
     )) AS duplicate_trusted_keys,

    (SELECT COUNT(*)
     FROM cityfix.cityfix.silver_trusted_requests
     WHERE dq_status <> 'PASS'
        OR route <> 'TRUSTED') AS invalid_trusted_routes;

candidate_rows,trusted_rows,open_quarantine_rows,historical_quarantine_rows,replayed_trusted_rows,null_trusted_keys,duplicate_trusted_keys,invalid_trusted_routes
180012,179701,311,312,1,0,0,0
